# Losslessness under pressure — cross-vendor replication

Runs the **exact pre-registered** losslessness experiment (same document, same 45-word summary,
same 8 questions, same deterministic grading as `scenario.json` / `runs.json` / `grade.py`) on an
**open, non-Claude** instruct model. This is the one run that breaks the paper's single-family
limitation: if a Llama/Qwen/Mistral responder also scores higher with the lossless reference than with
the lossy summary, the mechanism is not Claude-specific.

Set runtime to **GPU (T4)** → Run all. Send back `lossless_crossvendor.json`.

In [ ]:
!pip -q install -U transformers accelerate
import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Pick any open instruct model. Qwen needs no gated access; Llama-3.1-8B-Instruct needs an HF token.
MODEL = 'Qwen/Qwen2.5-7B-Instruct'   # or 'mistralai/Mistral-7B-Instruct-v0.3', 'meta-llama/Llama-3.1-8B-Instruct'
N_PER_ARM = 10                        # reviewer asked for n>=10

import json, torch, re
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map='auto').eval()
print('loaded', MODEL)

In [ ]:
# The pre-registered materials (identical to benchmark/ab_study/lossless/scenario.json).
DOCUMENT = (
  'Post-incident report, INC-7732. The checkout service degraded on 14 March between 09:12 and 09:58 '
  'UTC. Root cause: a config push set the database connection-pool ceiling to 25 (down from the '
  'intended 250), so the pool saturated under normal load. The on-call engineer, Priya Nair, rolled '
  'back the config at 09:51. Customer impact: 4,310 failed checkouts and an estimated 18,700 USD in '
  'lost revenue. A contributing factor was that the staging environment uses a different config '
  'schema, so the bad value passed staging validation. The fix shipped in PR #4471, which adds a hard '
  'lower bound of 100 on the pool ceiling and a schema-parity check between staging and production. '
  'One follow-up remains open: the alert for pool saturation fires at 95% utilisation, which the team '
  'agreed is too late and should move to 80%. The incident commander noted that the runbook link in '
  'the alert was stale and pointed to a deleted wiki page.')

# Same faithful 45-word summary used in the Claude run (drops the rollback time and the staging reason).
SUMMARY = (
  'Config push set the checkout DB connection-pool ceiling to 25 instead of 250, causing saturation '
  'and a 46-min degradation (4,310 failed checkouts, ~$18.7K lost) on 14 March. Priya rolled back; '
  'PR #4471 adds a 100 floor and staging/prod schema-parity check. Open: move saturation alert from '
  '95% to 80%; alert runbook link is stale.')

QUESTIONS = [
  ('What value was the database connection-pool ceiling mistakenly set to?', [['25']]),
  ('Who rolled back the config, and at what time?', [['priya', '09:51']]),
  ('How many failed checkouts occurred?', [['4,310'], ['4310']]),
  ('Which PR shipped the fix?', [['4471']]),
  ('What hard lower bound on the pool ceiling does the fix add?', [['100']]),
  ('Why did the bad config value pass staging validation?', [['different config schema'], ['different schema'], ['schema']]),
  ('What utilisation threshold should the saturation alert move to?', [['80']]),
  ('What was wrong with the runbook link in the alert?', [['stale'], ['deleted']]),
]

def correct(ans, groups):
    a = ans.lower()
    if 'not in context' in a:
        return 0
    # each group is a list of substrings that must ALL appear; any group satisfying => correct
    return 1 if any(all(s in a for s in g) for g in groups) else 0

In [ ]:
def ask(context):
    qs = '\n'.join(f'{i+1}. {q}' for i, (q, _) in enumerate(QUESTIONS))
    sys = ('Answer each question ONLY from the context. If the context does not contain the answer, '
           'reply exactly "NOT IN CONTEXT" for that question. Number your answers 1-8.')
    msg = [{'role': 'system', 'content': sys},
           {'role': 'user', 'content': f'Context:\n"{context}"\n\nQuestions:\n{qs}'}]
    ids = tok.apply_chat_template(msg, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(ids, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

def parse(text):
    # split into 8 numbered answers; tolerant of formatting
    parts = re.split(r'\n?\s*([1-8])[.)]\s', '\n' + text)
    ans = {}
    for i in range(1, len(parts) - 1, 2):
        try:
            ans[int(parts[i])] = parts[i + 1].strip()
        except (ValueError, IndexError):
            pass
    return [ans.get(i + 1, '') for i in range(8)]

import statistics
samples = []
for arm, ctx in [('summary', SUMMARY), ('reference', DOCUMENT)]:
    for r in range(N_PER_ARM):
        raw = ask(ctx)
        answers = parse(raw)
        score = sum(correct(answers[i], g) for i, (_, g) in enumerate(QUESTIONS))
        samples.append({'arm': arm, 'run': r, 'score': score, 'answers': answers})
        print(f'{arm:9} run {r}: {score}/8')

summ = [s['score'] for s in samples if s['arm'] == 'summary']
ref  = [s['score'] for s in samples if s['arm'] == 'reference']
result = {'model': MODEL, 'n_per_arm': N_PER_ARM,
          'summary_mean': statistics.mean(summ), 'reference_mean': statistics.mean(ref),
          'summary_scores': summ, 'reference_scores': ref, 'samples': samples}
json.dump(result, open('lossless_crossvendor.json', 'w'), indent=2)
print(f'\n{MODEL}: summary {statistics.mean(summ):.2f}/8  vs  reference {statistics.mean(ref):.2f}/8')

In [ ]:
from google.colab import files
files.download('lossless_crossvendor.json')